# ForkWise Full Stack Provisioner

**Zero assumptions.** This notebook installs every dependency it needs.

Run cells **top to bottom**. Each cell is re-runnable on failure.

**You provide only:**
1. `clouds.yaml` — copied into `infra/tf/kvm/` before step 4
2. OpenStack object-store access key
3. OpenStack object-store secret key
4. GHCR token (optional — blank if images are public)

## 0. Install System Dependencies

Installs Terraform, Ansible, kubectl, and ssh client into this Jupyter environment.

In [ ]:
import subprocess, sys, os

def sh(cmd, check=True):
    print(f">>> {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Failed: {cmd}")
    return r

# ── Terraform ────────────────────────────────────────────────────────
tf_check = subprocess.run("which terraform", shell=True, capture_output=True)
if tf_check.returncode != 0:
    print("Installing Terraform...")
    sh("apt-get update -qq && apt-get install -y -qq gnupg software-properties-common curl")
    sh("curl -fsSL https://apt.releases.hashicorp.com/gpg | gpg --dearmor -o /usr/share/keyrings/hashicorp.gpg")
    sh('echo "deb [signed-by=/usr/share/keyrings/hashicorp.gpg] https://apt.releases.hashicorp.com $(lsb_release -cs) main" > /etc/apt/sources.list.d/hashicorp.list')
    sh("apt-get update -qq && apt-get install -y -qq terraform")
else:
    print("Terraform already installed")

# ── Ansible ──────────────────────────────────────────────────────────
ans_check = subprocess.run("which ansible-playbook", shell=True, capture_output=True)
if ans_check.returncode != 0:
    print("Installing Ansible...")
    sh(f"{sys.executable} -m pip install --quiet ansible")
else:
    print("Ansible already installed")

# ── kubectl ──────────────────────────────────────────────────────────
kc_check = subprocess.run("which kubectl", shell=True, capture_output=True)
if kc_check.returncode != 0:
    print("Installing kubectl...")
    sh("curl -LO 'https://dl.k8s.io/release/$(curl -Ls https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl'")
    sh("install -o root -g root -m 0755 kubectl /usr/local/bin/kubectl && rm kubectl")
else:
    print("kubectl already installed")

# ── SSH client ───────────────────────────────────────────────────────
ssh_check = subprocess.run("which ssh", shell=True, capture_output=True)
if ssh_check.returncode != 0:
    print("Installing openssh-client...")
    sh("apt-get update -qq && apt-get install -y -qq openssh-client")
else:
    print("SSH client already installed")

# ── python-openstackclient (for clouds.yaml auth) ────────────────────
sh(f"{sys.executable} -m pip install --quiet python-openstackclient", check=False)

print("\n\u2713 All system dependencies installed.")

## 1. Clone Repo

Clones the repo if not already present. Sets working directory.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/HivanshD/ml-sys-ops-project.git"
REPO_DIR = Path.home() / "ml-sys-ops-project"

if not (REPO_DIR / ".git").exists():
    sh(f"git clone {REPO_URL} {REPO_DIR}")
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

## 2. Setup Helpers & Constants

In [ ]:
import subprocess, sys, os, textwrap, time
from datetime import datetime
from pathlib import Path
from IPython.display import display, HTML

REPO_ROOT   = Path.cwd()
TF_DIR      = REPO_ROOT / "infra" / "tf" / "kvm"
ANSIBLE_DIR = REPO_ROOT / "infra" / "ansible"
K8S_DIR     = REPO_ROOT / "infra" / "k8s"
LOG_DIR     = REPO_ROOT / "provision_logs"
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE    = LOG_DIR / f"provision_{datetime.now():%Y%m%d_%H%M%S}.log"
SESSION_FILE = REPO_ROOT / "SESSION_INFO.txt"
_log_fh = open(LOG_FILE, "a")

SUFFIX           = "proj01"
KEYPAIR_NAME     = "id_rsa"
SSH_KEY_PATH     = "/work/.ssh/id_rsa"
IMAGE_NAME       = "CC-Ubuntu24.04"
FLAVOR_NAME      = "m1.medium"
SHAREDNET        = "sharednet1"
FIP_POOL         = "public"
SECURITY_GROUPS  = '["default"]'
SERVING_IMAGE    = "ghcr.io/itsnotaka/subst-serving-onnx:demo"
AUTOMATION_IMAGE = "ghcr.io/itsnotaka/forkwise-automation:demo"
GHCR_USER        = "itsnotaka"
OS_ENDPOINT      = "https://chi.tacc.chameleoncloud.org:7480"
NODES = {"node1": "192.168.1.11", "node2": "192.168.1.12", "node3": "192.168.1.13"}

FLOATING_IP = ""
SESSION_LINES = []

def _c(colour, msg):
    cm = {"green": "#2ecc71", "red": "#e74c3c", "yellow": "#f39c12", "blue": "#3498db"}
    return f'<span style="color:{cm.get(colour,"#ccc")}">{msg}</span>'

def log(msg, level="INFO"):
    ts = datetime.now().strftime("%H:%M:%S")
    c = {"INFO":"blue","OK":"green","WARN":"yellow","FAIL":"red"}.get(level,"blue")
    display(HTML(f'<code>{_c(c,f"[{ts}] [{level}]")} {msg}</code>'))
    _log_fh.write(f"[{ts}] [{level}] {msg}\n"); _log_fh.flush()

def banner(step, title):
    display(HTML(f'<h3 style="color:#3498db">Step {step}: {title}</h3>'))

def run(cmd, cwd=None, env=None, check=True, timeout=600):
    log(f"$ {cmd}")
    merged = {**os.environ, **(env or {})}
    r = subprocess.run(cmd, shell=True, cwd=cwd, env=merged, capture_output=True, text=True, timeout=timeout)
    _log_fh.write(r.stdout + "\n")
    if r.stderr: _log_fh.write(r.stderr + "\n")
    if r.stdout.strip():
        lines = r.stdout.strip().split("\n")[-40:]
        display(HTML(f'<pre style="font-size:11px;max-height:300px;overflow:auto">{"<br>".join(lines)}</pre>'))
    if check and r.returncode != 0:
        log(f"FAILED (rc={r.returncode}): {r.stderr.strip()[-500:]}", "FAIL")
        raise RuntimeError(f"Command failed: {cmd}")
    return r

def record(label, value):
    SESSION_LINES.append(f"{label}: {value}")
    log(f"\u2192 {label}: {value}", "OK")

log("Helpers loaded", "OK")
log(f"Log file: {LOG_FILE}", "INFO")

## 3. Collect Secrets

In [ ]:
import getpass

OS_ACCESS_KEY = input("OpenStack object-store access key: ").strip()
OS_SECRET_KEY = getpass.getpass("OpenStack object-store secret key: ").strip()
GHCR_TOKEN    = getpass.getpass("GHCR token (blank if public): ").strip()

assert OS_ACCESS_KEY and OS_SECRET_KEY, "Object-store keys are required"
log("Secrets collected", "OK")

## 4. Setup SSH Agent

Starts ssh-agent and loads the keypair for Ansible to reach the VMs.

In [ ]:
banner(4, "Setup SSH Agent")

# Check key exists
key_path = Path(SSH_KEY_PATH)
if not key_path.exists():
    log(f"SSH key not found at {SSH_KEY_PATH}", "FAIL")
    log("Upload your private key or update SSH_KEY_PATH in cell 2.", "FAIL")
    raise FileNotFoundError(SSH_KEY_PATH)

# Fix permissions
sh(f"chmod 600 {SSH_KEY_PATH}", check=False)

# Start agent and add key
import tempfile, re
r = subprocess.run("ssh-agent -s", shell=True, capture_output=True, text=True)
for line in r.stdout.split(";"):
    m = re.match(r"(SSH_AUTH_SOCK|SSH_AGENT_PID)=(.+)", line.strip())
    if m:
        os.environ[m.group(1)] = m.group(2)

sh(f"ssh-add {SSH_KEY_PATH}")
log("SSH agent running, key loaded", "OK")

## 5. Provision VMs (Terraform)\n\n**Prerequisite:** `clouds.yaml` must exist in `infra/tf/kvm/`.

In [ ]:
banner(5, "Provision VMs via Terraform")

tfvars_content = f"""suffix               = "{SUFFIX}"
key                  = "{KEYPAIR_NAME}"
image_name           = "{IMAGE_NAME}"
flavor_id            = ""
flavor_name          = "{FLAVOR_NAME}"
sharednet_name       = "{SHAREDNET}"
floating_ip_pool     = "{FIP_POOL}"
security_group_names = {SECURITY_GROUPS}
"""
(TF_DIR / "terraform.tfvars").write_text(tfvars_content)
log("Wrote terraform.tfvars", "OK")

clouds = TF_DIR / "clouds.yaml"
if not clouds.exists():
    log("clouds.yaml not found in infra/tf/kvm/", "FAIL")
    log("Copy your clouds.yaml there before running this cell.", "FAIL")
    raise FileNotFoundError("clouds.yaml")

tf_env = {"OS_CLIENT_CONFIG_FILE": str(clouds), "OS_CLOUD": "openstack"}

run("terraform init",                 cwd=TF_DIR, env=tf_env)
run("terraform validate",             cwd=TF_DIR, env=tf_env)
run("terraform apply -auto-approve",  cwd=TF_DIR, env=tf_env, timeout=900)

r = run("terraform output -raw floating_ip", cwd=TF_DIR, env=tf_env)
FLOATING_IP = r.stdout.strip()
record("Floating IP", FLOATING_IP)
record("Private IPs", "node1=192.168.1.11  node2=192.168.1.12  node3=192.168.1.13")

### 5b. Override Floating IP *(skip if step 5 succeeded)*

In [ ]:
# FLOATING_IP = "x.x.x.x"   # <-- uncomment and set if resuming
assert FLOATING_IP, "Set FLOATING_IP above or run step 5"
log(f"Floating IP = {FLOATING_IP}", "OK")

## 6. Bootstrap k3s (Ansible)

In [ ]:
banner(6, "Bootstrap k3s")
assert FLOATING_IP

ansible_cfg = f"""[defaults]
inventory = ./inventory.yml
stdout_callback = default
callback_result_format = yaml
host_key_checking = False
retry_files_enabled = False

[ssh_connection]
ssh_args = -o ProxyJump=cc@{FLOATING_IP} -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ControlMaster=auto -o ControlPersist=60s
pipelining = True
"""
(ANSIBLE_DIR / "ansible.cfg").write_text(ansible_cfg)
log(f"ansible.cfg -> {FLOATING_IP}", "OK")

for pb, desc, tout in [
    ("general/hello_host.yml",          "Connectivity check", 120),
    ("pre_k8s/pre_k8s_configure.yml",   "OS-level prep",      300),
    ("k8s/install_k3s.yml",             "k3s install",        600),
    ("post_k8s/post_k8s_configure.yml", "Post-k3s config",    300),
]:
    log(f"Running: {desc}")
    run(f"ansible-playbook -i inventory.yml {pb}", cwd=ANSIBLE_DIR, timeout=tout)
    log(f"\u2713 {desc}", "OK")

r = run(f"ssh -o StrictHostKeyChecking=no cc@{FLOATING_IP} 'kubectl get nodes -o wide'", check=False)
if r.returncode == 0:
    record("k3s cluster", "3 nodes ready")
else:
    log("kubectl get nodes failed \u2014 check SSH", "WARN")

## 7. GHCR Docker Login

In [ ]:
banner(7, "GHCR Docker Login")
if not GHCR_TOKEN:
    log("No GHCR token \u2014 assuming public images, skipping.", "WARN")
else:
    run(f"echo '{GHCR_TOKEN}' | docker login ghcr.io -u {GHCR_USER} --password-stdin")
    log("GHCR login \u2713", "OK")

## 8. Deploy Mealie + Serving

In [ ]:
banner(8, "Deploy Mealie + Serving")
assert FLOATING_IP

run(
    f"ansible-playbook -i inventory.yml deploy/deploy_apps.yml "
    f"-e serving_image={SERVING_IMAGE}",
    cwd=ANSIBLE_DIR, timeout=600,
)
record("Mealie",  f"ssh -N -L 9000:127.0.0.1:30090 cc@{FLOATING_IP}  -> http://localhost:9000")
record("Serving", f"ssh -N -L 8000:127.0.0.1:30080 cc@{FLOATING_IP}  -> http://localhost:8000/health")

## 9. Create K8s Secrets

In [ ]:
banner(9, "Create K8s Secrets")
assert FLOATING_IP
ssh = f"ssh -o StrictHostKeyChecking=no cc@{FLOATING_IP}"

for ns in ["monitoring-proj01", "staging-proj01", "canary-proj01", "production-proj01"]:
    run(f"{ssh} 'kubectl create namespace {ns} --dry-run=client -o yaml | kubectl apply -f - && "
        f"kubectl create secret generic os-credentials -n {ns} "
        f"--from-literal=OS_ENDPOINT={OS_ENDPOINT} "
        f"--from-literal=OS_ACCESS_KEY={OS_ACCESS_KEY} "
        f"--from-literal=OS_SECRET_KEY={OS_SECRET_KEY} "
        f"--dry-run=client -o yaml | kubectl apply -f -'", check=False)

for ns in ["forkwise-data", "forkwise-serving"]:
    run(f"{ssh} 'kubectl create namespace {ns} --dry-run=client -o yaml | kubectl apply -f - && "
        f"kubectl create secret generic s3-credentials -n {ns} "
        f"--from-literal=access-key={OS_ACCESS_KEY} "
        f"--from-literal=secret-key={OS_SECRET_KEY} "
        f"--dry-run=client -o yaml | kubectl apply -f -'", check=False)

if GHCR_TOKEN:
    run(f"{ssh} 'kubectl create secret docker-registry ghcr-pull -n forkwise-data "
        f"--docker-server=ghcr.io --docker-username={GHCR_USER} --docker-password={GHCR_TOKEN} "
        f"--dry-run=client -o yaml | kubectl apply -f -'", check=False)

log("All secrets created \u2713", "OK")

## 10. Deploy Rollout Stack

In [ ]:
banner(10, "Deploy Rollout Stack")
assert FLOATING_IP

run(
    f"ansible-playbook -i inventory.yml deploy/deploy_rollout_stack.yml "
    f"-e serving_image={SERVING_IMAGE} "
    f"-e automation_image={AUTOMATION_IMAGE}",
    cwd=ANSIBLE_DIR, timeout=900,
)
record("Grafana",    f"ssh -N -L 3000:127.0.0.1:30300 cc@{FLOATING_IP}  -> http://localhost:3000")
record("Prometheus", f"ssh -N -L 9090:127.0.0.1:30900 cc@{FLOATING_IP}  -> http://localhost:9090")

## 11. Deploy Data Workloads + Ingest

In [ ]:
banner(11, "Deploy Data Workloads")
assert FLOATING_IP
ssh = f"ssh -o StrictHostKeyChecking=no cc@{FLOATING_IP}"

run(f"{ssh} 'kubectl apply -k /tmp/serving-k8s/apps/forkwise-data'", check=False)

log("Running ingest job (may take several minutes)...")
run(f"{ssh} 'kubectl apply -f /tmp/serving-k8s/apps/forkwise-data/job-ingest.yaml'", check=False)
run(f"{ssh} 'kubectl wait --for=condition=complete job/forkwise-ingest -n forkwise-data --timeout=1800s'",
    check=False, timeout=1860)

run(f"{ssh} 'kubectl scale deployment/data-generator -n forkwise-data --replicas=1'", check=False)

patch = '{\"spec\":{\"suspend\":false}}'
run(f"{ssh} \"kubectl patch cronjob batch-pipeline -n forkwise-data -p '{patch}'\"", check=False)
run(f"{ssh} \"kubectl patch cronjob drift-monitor -n forkwise-data -p '{patch}'\"", check=False)

log("Data workloads enabled \u2713", "OK")

## 12. Smoke Test

In [ ]:
banner(12, "Smoke Test")
assert FLOATING_IP
ssh = f"ssh -o StrictHostKeyChecking=no cc@{FLOATING_IP}"

for cmd, desc in [
    ("kubectl get nodes",       "Cluster nodes"),
    ("kubectl get pods -A",     "All pods"),
    ("curl -sf http://127.0.0.1:30080/health || echo 'not ready'", "Serving health"),
    ("curl -sf http://127.0.0.1:30090 -o /dev/null && echo 'ok' || echo 'not ready'", "Mealie health"),
]:
    log(f"Check: {desc}")
    r = run(f"{ssh} '{cmd}'", check=False)
    log(f"{'\u2713' if r.returncode == 0 else '\u2717'} {desc}", "OK" if r.returncode == 0 else "WARN")

## 13. Session Info *(save this)*

In [ ]:
info = f"""
======================================================================
  FORKWISE SESSION \u2014 {datetime.now():%Y-%m-%d %H:%M:%S}
======================================================================

SSH ACCESS
  Control:  ssh cc@{FLOATING_IP}
  Worker2:  ssh -J cc@{FLOATING_IP} cc@192.168.1.12
  Worker3:  ssh -J cc@{FLOATING_IP} cc@192.168.1.13

SERVICE TUNNELS
  Mealie:      ssh -N -L 9000:127.0.0.1:30090 cc@{FLOATING_IP}  -> http://localhost:9000
  Serving:     ssh -N -L 8000:127.0.0.1:30080 cc@{FLOATING_IP}  -> http://localhost:8000/health
  Grafana:     ssh -N -L 3000:127.0.0.1:30300 cc@{FLOATING_IP}  -> http://localhost:3000
  Prometheus:  ssh -N -L 9090:127.0.0.1:30900 cc@{FLOATING_IP}  -> http://localhost:9090

IMAGES
  Serving:     {SERVING_IMAGE}
  Automation:  {AUTOMATION_IMAGE}

LOG: {LOG_FILE}
"""

if SESSION_LINES:
    info += "\nRECORDED\n"
    for line in SESSION_LINES:
        info += f"  {line}\n"

SESSION_FILE.write_text(info)
display(HTML(f'<pre style="background:#1a1a2e;color:#e0e0e0;padding:16px;border-radius:8px;font-size:12px">{info}</pre>'))
log(f"Saved to {SESSION_FILE}", "OK")